# Stages 5, 12, 13 — GPT-5.5 (low) vs Qwen+v2 LoRA vs Qwen base, one Run All

Three-model head-to-head on the three **scored** non-detection stages:

| Stage | Task | Ground truth |
|---|---|---|
| 5 OCR tiebreaker | true word vs 1-char-corrupted word, given the crop | pseudo-GT (Tesseract conf>=70) — real accuracy on a controlled task |
| 12 Relation validation | accept/reject injected true/false connections | **real (PID2Graph)** — the most trustworthy number, and exactly what the v2 adapter trained on |
| 13 Entity validation | keep/remove: real symbol crop vs empty-region decoy | partial (Gupta, existence-only) |

Stages 1 and 2 are excluded on purpose: they have no ground truth, so they cannot rank models.

**Design:** pools are built ONCE, then all three models answer the identical examples —
GPT-5.5 via API, then Qwen base (loaded once), then the v2 LoRA attached to the same weights
in-place (no second model load). Needs a GPU (A100 fine) + your OpenAI key.
Cost: ~150 GPT-5.5 vision calls.

## 1. Config

In [ ]:
HF_TOKEN = "paste-your-hf-token-here"
OPENAI_API_KEY = "paste-your-openai-key-here"

GPT_REASONING_EFFORT = "low"
ADAPTER_PATH_IN_REPO = "v2/latest"     # the v2 LoRA
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
N_PER_STAGE = 25

DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
RESULTS_PATH_IN_REPO = "benchmarks/three_stage_head2head.csv"

assert HF_TOKEN.startswith("hf_") and OPENAI_API_KEY.startswith("sk-")

## 2. Install + data (Gupta fixtures via GitHub; PID2Graph via HF)

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract huggingface_hub openai peft
!pip uninstall -y torchao -q

import zipfile, time, json, random, re, io, base64, csv, datetime
from pathlib import Path
from huggingface_hub import hf_hub_download

SHEETS = Path("/content/gupta_test/sheets"); SHEETS.mkdir(parents=True, exist_ok=True)
LABELS = Path("/content/gupta_test/labels"); LABELS.mkdir(parents=True, exist_ok=True)
BASE = "https://raw.githubusercontent.com/TomGeorge45/pid-opensrc-substitution/main/notebooks/stage4/gupta_test_sheets"
TEST_IDS = ["0","103","11","124","129","136","145","148","15","151",
            "157","158","159","176","188","192","194","196","216","233"]
for sid in TEST_IDS:
    !wget -q -O {SHEETS}/{sid}.jpg {BASE}/sheets/{sid}.jpg
    !wget -q -O {LABELS}/{sid}.txt {BASE}/labels/{sid}.txt
for sid in TEST_IDS:
    assert (SHEETS/f"{sid}.jpg").stat().st_size > 1000, sid

P2G = Path("/content/data/pid2graph")
if not (P2G.exists() and any(P2G.iterdir())):
    zp = hf_hub_download(repo_id=DATA_REPO, filename="pid2graph/PID2Graph.zip",
                         repo_type="dataset", token=HF_TOKEN)
    P2G.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(P2G)
PID2GRAPH_OPEN100 = P2G / "PID2Graph" / "Patched" / "PID2Graph OPEN100"
assert PID2GRAPH_OPEN100.exists()
print("data ready")

## 3. Build the three eval pools ONCE (identical examples for all models)

In [ ]:
import xml.etree.ElementTree as ET
import pytesseract
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
rng = random.Random(42)
TEST_SHEETS = sorted(SHEETS.glob("*.jpg"))

def yolo_boxes(label_path, W, H):
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        if line.strip():
            parts = line.split()
            cx, cy, w, h = (float(v) for v in parts[1:5])
            boxes.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H])
    return boxes

def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

# --- Stage 5 pool: true vs corrupted word ---
def corrupt(word):
    i = rng.randrange(len(word))
    repl = {"0":"8","1":"7","5":"6","B":"8","O":"0","I":"1"}.get(
        word[i], rng.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-"))
    while repl == word[i]:
        repl = rng.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-")
    return word[:i] + repl + word[i+1:]

stage5_pool = []
for sheet in TEST_SHEETS:
    if len(stage5_pool) >= N_PER_STAGE: break
    img = Image.open(sheet).convert("RGB")
    tile = img.crop((0, 0, min(2048, img.width), min(2048, img.height)))
    data = pytesseract.image_to_data(tile, output_type=pytesseract.Output.DICT)
    for i in range(len(data["text"])):
        t = data["text"][i].strip()
        if (len(t) >= 4 and int(data["conf"][i]) >= 70
                and (any(c.isdigit() for c in t) or "-" in t) and len(stage5_pool) < N_PER_STAGE):
            m = 20
            crop = tile.crop((max(0, data["left"][i]-m), max(0, data["top"][i]-m),
                              data["left"][i]+data["width"][i]+m, data["top"][i]+data["height"][i]+m))
            a_is_true = rng.random() < 0.5
            A, B = (t, corrupt(t)) if a_is_true else (corrupt(t), t)
            stage5_pool.append({"crop": crop, "a_is_true": a_is_true,
                "prompt": (f"This crop shows one text label from an engineering drawing. Two OCR "
                           f"readings were proposed:\nA: {A}\nB: {B}\nWhich exactly matches the "
                           f"pixels? Answer A or B only.")})

# --- Stage 12 pool: relation accept/reject (real PID2Graph GT) ---
MAX_SPAN, MARGIN = 1400, 80
stage12_pool = []
gmls = sorted(PID2GRAPH_OPEN100.rglob("*.graphml"))
rng.shuffle(gmls)
for gml in gmls:
    if len(stage12_pool) >= N_PER_STAGE: break
    png = gml.with_suffix(".png")
    if not png.exists(): continue
    try:
        nodes, edges = parse_graphml(gml)
    except ET.ParseError:
        continue
    if len(nodes) < 4 or not edges: continue
    want = len(stage12_pool) % 2 == 0
    pair = None
    if want:
        pool = [tuple(e) for e in edges]
        pair = pool[rng.randrange(len(pool))]
    else:
        for _ in range(50):
            a, b = rng.sample(list(nodes), 2)
            if frozenset((a, b)) not in edges:
                pair = (a, b); break
    if not pair: continue
    a, b = pair
    ba, bb = nodes[a], nodes[b]
    ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
    ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
    if ux1-ux0 > MAX_SPAN or uy1-uy0 > MAX_SPAN: continue
    cx0, cy0 = max(0, int(ux0-MARGIN)), max(0, int(uy0-MARGIN))
    img = Image.open(png).convert("RGB")
    crop = img.crop((cx0, cy0, int(ux1+MARGIN), int(uy1+MARGIN)))
    stage12_pool.append({"crop": crop, "connected": want,
        "prompt": (f"In this P&ID crop there is a symbol at "
                   f"[{int(ba[0]-cx0)}, {int(ba[1]-cy0)}, {int(ba[2]-cx0)}, {int(ba[3]-cy0)}] and another at "
                   f"[{int(bb[0]-cx0)}, {int(bb[1]-cy0)}, {int(bb[2]-cx0)}, {int(bb[3]-cy0)}] "
                   f"(pixel coordinates, top-left origin). Are these two symbols directly connected "
                   f"to each other? Answer yes or no.")})

# --- Stage 13 pool: keep/remove (real symbol vs empty decoy) ---
def overlaps_any(box, boxes):
    return any(box[0] < b[2] and box[2] > b[0] and box[1] < b[3] and box[3] > b[1] for b in boxes)

stage13_pool = []
for sheet in TEST_SHEETS:
    if len(stage13_pool) >= N_PER_STAGE: break
    img = Image.open(sheet).convert("RGB")
    W, H = img.size
    boxes = yolo_boxes(LABELS/f"{sheet.stem}.txt", W, H)
    if not boxes: continue
    want_real = len(stage13_pool) % 2 == 0
    if want_real:
        b = boxes[rng.randrange(len(boxes))]
    else:
        b = None
        for _ in range(80):
            size = rng.uniform(30, 80)
            x0 = rng.uniform(0, W-size); y0 = rng.uniform(0, H-size)
            cand = [x0, y0, x0+size, y0+size]
            if not overlaps_any(cand, boxes):
                b = cand; break
        if b is None: continue
    m = 60
    crop = img.crop((max(0, int(b[0]-m)), max(0, int(b[1]-m)),
                     min(W, int(b[2]+m)), min(H, int(b[3]+m))))
    cb = [int(b[0]-max(0, b[0]-m)), int(b[1]-max(0, b[1]-m)),
          int(b[2]-max(0, b[0]-m)), int(b[3]-max(0, b[1]-m))]
    stage13_pool.append({"crop": crop, "is_real": want_real,
        "prompt": (f"A symbol detector claims there is a P&ID equipment symbol at "
                   f"[{cb[0]}, {cb[1]}, {cb[2]}, {cb[3]}] in this crop (pixel coords, top-left origin). "
                   f"Verify against the pixels: is there really a discrete equipment symbol there "
                   f"(valve, instrument, fitting)? Answer keep or remove.")})

print(f"pools: stage5={len(stage5_pool)} stage12={len(stage12_pool)} stage13={len(stage13_pool)}")

## 4. Scoring (shared by all models)

In [ ]:
def eval_model(generate, label):
    """generate(pil_image, prompt) -> text. Returns {stage: metrics}."""
    res = {}
    t0 = time.time()
    # stage 5
    correct = undecided = 0
    for ex in stage5_pool:
        ans = generate(ex["crop"], ex["prompt"]).strip().upper()
        if not ans or ans[0] not in "AB":
            undecided += 1; continue
        correct += (ans[0] == "A") == ex["a_is_true"]
    n_dec = len(stage5_pool) - undecided
    res["5_ocr_tiebreak"] = {"acc_%": round(100*correct/n_dec, 1) if n_dec else None,
                             "undecided": undecided, "n": len(stage5_pool)}
    print(f"  [{label}] stage5 done {time.time()-t0:.0f}s")
    # stage 12
    correct = undecided = 0
    for ex in stage12_pool:
        ans = generate(ex["crop"], ex["prompt"]).lower()
        yes, no = ans.startswith("yes"), ans.startswith("no")
        if not yes and not no:
            undecided += 1; continue
        correct += (yes == ex["connected"])
    n_dec = len(stage12_pool) - undecided
    res["12_relation"] = {"acc_%": round(100*correct/n_dec, 1) if n_dec else None,
                          "undecided": undecided, "n": len(stage12_pool)}
    print(f"  [{label}] stage12 done {time.time()-t0:.0f}s")
    # stage 13
    correct = undecided = 0
    for ex in stage13_pool:
        ans = generate(ex["crop"], ex["prompt"]).lower()
        keep, remove = ans.startswith("keep"), ans.startswith("remove")
        if not keep and not remove:
            undecided += 1; continue
        correct += (keep == ex["is_real"])
    n_dec = len(stage13_pool) - undecided
    res["13_entity_val"] = {"acc_%": round(100*correct/n_dec, 1) if n_dec else None,
                            "undecided": undecided, "n": len(stage13_pool)}
    print(f"  [{label}] stage13 done {time.time()-t0:.0f}s")
    return res

all_results = {}

## 5. Model 1 — GPT-5.5 (low reasoning, API)

In [ ]:
from openai import OpenAI
_oa = OpenAI(api_key=OPENAI_API_KEY)

def gpt_generate(image, prompt, max_tokens=64):
    buf = io.BytesIO(); image.save(buf, format="PNG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode()
    resp = _oa.chat.completions.create(
        model="gpt-5.5", reasoning_effort=GPT_REASONING_EFFORT, max_completion_tokens=max_tokens,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": prompt},
        ]}])
    return (resp.choices[0].message.content or "").strip()

all_results["gpt-5.5-low"] = eval_model(gpt_generate, "gpt-5.5-low")

In [ ]:
# --- resilient GPT stage-13 retry: keeps stage 5/12 results already in all_results ---
_first_err = [None]

def gpt_generate_safe(image, prompt, max_tokens=64):
    try:
        return gpt_generate(image, prompt, max_tokens)
    except Exception as e:
        if _first_err[0] is None:
            _first_err[0] = e
            body = getattr(e, "body", None) or getattr(e, "message", None) or str(e)
            print(f"[first error detail] {type(e).__name__}: {body}")
            print(f"[offending image size] {image.size}")
        return ""   # counts as undecided, run continues

correct = undecided = 0
for ex in stage13_pool:
    ans = gpt_generate_safe(ex["crop"], ex["prompt"]).lower()
    keep, remove = ans.startswith("keep"), ans.startswith("remove")
    if not keep and not remove:
        undecided += 1
        continue
    correct += (keep == ex["is_real"])

n_dec = len(stage13_pool) - undecided
all_results["gpt-5.5-low"]["13_entity_val"] = {
    "acc_%": round(100*correct/n_dec, 1) if n_dec else None,
    "undecided": undecided, "n": len(stage13_pool)}
print(all_results["gpt-5.5-low"])

In [ ]:
all_results["gpt-5.5-low"] = eval_model(gpt_generate_safe, "gpt-5.5-low")

## 6. Model 2 — Qwen base (no adapter)

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()

def qwen_generate(image, prompt, max_tokens=32):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(qwen.device)
    with torch.no_grad():
        out = qwen.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

all_results["qwen-base"] = eval_model(qwen_generate, "qwen-base")

## 7. Model 3 — Qwen + v2 LoRA (attached in-place, no reload)

In [ ]:
from huggingface_hub import snapshot_download
from peft import PeftModel

local = Path("/content/h2h_ckpt")
snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                  allow_patterns=[f"{ADAPTER_PATH_IN_REPO}/*"], local_dir=str(local))
adapter_dir = local / ADAPTER_PATH_IN_REPO
assert (adapter_dir / "adapter_model.safetensors").exists(), f"no adapter at {adapter_dir}"

qwen = PeftModel.from_pretrained(qwen, str(adapter_dir))
qwen.eval()
print(f"v2 adapter attached from {CKPT_REPO}/{ADAPTER_PATH_IN_REPO}")

all_results["qwen-domain-v2"] = eval_model(qwen_generate, "qwen-domain-v2")

## 8. Verdict table + push to HF

In [ ]:
models = list(all_results)
stages = ["5_ocr_tiebreak", "12_relation", "13_entity_val"]
print(f"{'stage':16s}" + "".join(f"{m:>18s}" for m in models) + "   baseline")
print("-" * (16 + 18*len(models) + 11))
for st in stages:
    line = f"{st:16s}"
    for m in models:
        r = all_results[m][st]
        acc = r["acc_%"]
        line += f"{(str(acc)+'%' if acc is not None else 'n/a'):>13s} (u{r['undecided']})"
    line += "      50%"
    print(line)
print("\nRead with the honesty table: 12 = real GT (the number that matters most, and the v2 "
      "adapter's home turf); 5 = pseudo-GT controlled task; 13 = existence-only.")

RESULTS_LOCAL = Path("/content/three_stage_head2head.csv")
FIELDS = ["timestamp","model","stage","acc_%","undecided","n","notes"]
try:
    prev = hf_hub_download(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                           repo_type="dataset", token=HF_TOKEN)
    RESULTS_LOCAL.write_text(Path(prev).read_text())
except Exception:
    with open(RESULTS_LOCAL, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()
with open(RESULTS_LOCAL, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for m in models:
        for st in stages:
            r = all_results[m][st]
            w.writerow({"timestamp": ts, "model": m, "stage": st, "acc_%": r["acc_%"],
                        "undecided": r["undecided"], "n": r["n"], "notes": ""})

from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=str(RESULTS_LOCAL),
    path_in_repo=RESULTS_PATH_IN_REPO, repo_id=DATA_REPO, repo_type="dataset")
print("\nresults pushed to HF.")

In [ ]:
# ---------- FIX 1: GPT-5.5 with a real token budget (reasoning needs headroom) ----------
def gpt_generate_fixed(image, prompt, max_tokens=2000):
    return gpt_generate_safe(image, prompt, max_tokens)

all_results["gpt-5.5-low"] = eval_model(gpt_generate_fixed, "gpt-5.5-low-fixed")

# ---------- FIX 2: adapter-accommodated prompts (v2 stays attached) ----------
def norm(s):
    return re.sub(r'[^A-Z0-9]', '', s.upper())

res = {}
# stage 5: read-then-match (uses the adapter's trained reading template)
correct = undecided = 0
for ex in stage5_pool:
    m = re.search(r"A: (.+)\nB: (.+)\nWhich", ex["prompt"])
    A, B = m.group(1), m.group(2)
    ans = norm(qwen_generate(ex["crop"], "What does the text label in this crop say?", 32))
    sa = difflib.SequenceMatcher(None, ans, norm(A)).ratio()
    sb = difflib.SequenceMatcher(None, ans, norm(B)).ratio()
    if sa == sb:
        undecided += 1; continue
    correct += (sa > sb) == ex["a_is_true"]
n_dec = len(stage5_pool) - undecided
res["5_ocr_tiebreak"] = {"acc_%": round(100*correct/n_dec,1) if n_dec
                         "undecided": undecided, "n": len(stage5_pool)}

# stage 12: unchanged (already works)
res["12_relation"] = all_results["qwen-domain-v2"]["12_relation"]

# stage 13: yes/no phrasing (the adapter's trained answer format)
correct = undecided = 0
for ex in stage13_pool:
    p = ex["prompt"].replace("Answer keep or remove.", "Answer yes or no.")
    ans = qwen_generate(ex["crop"], p, 32).lower()
    yes, no = ans.startswith("yes"), ans.startswith("no")
    if not yes and not no:
        undecided += 1; continue
    correct += (yes == ex["is_real"])
n_dec = len(stage13_pool) - undecided
res["13_entity_val"] = {"acc_%": round(100*correct/n_dec,1) if n_dec
                        "undecided": undecided, "n": len(stage13_pool)}

all_results["qwen-domain-v2-tolerant"] = res

# ---------- reprint the verdict table ----------
models = list(all_results)
stages = ["5_ocr_tiebreak", "12_relation", "13_entity_val"]
header = f"{'stage':16s}" + "".join(f"{m:>28s}" for m in models) + "   baseline"
print(header)
print("-" * len(header))
for st in stages:
    line = f"{st:16s}"
    for m in models:
        r = all_results[m][st]
        acc = r["acc_%"]
        cell = (str(acc) + "%" if acc is not None else "n/a") + f" (u{r['undecided']})"
        line += f"{cell:>28s}"
    line += "      50%"
    print(line)

## 9. Fix round — GPT token budget + adapter-accommodated prompts

Two repairs, both harness-side (no retraining):
- **GPT-5.5 token starvation:** reasoning tokens count against `max_completion_tokens`;
  at 32-64 the visible answer often came back empty -> scored "undecided". Re-run with
  a 2000-token budget.
- **v2 adapter format lockout:** it can't emit "A"/"B" or "keep"/"remove", but it CAN
  answer yes/no (trained relation format) and read a label (trained reading format).
  Stage 13 -> yes/no phrasing; stage 5 -> read-then-match scoring. Labeled
  `qwen-domain-v2-tolerant` because the harness meets the model halfway - honest, but
  not like-for-like with the strict rows.

In [ ]:
import difflib

def gpt_generate_fixed(image, prompt, max_tokens=2000):
    """Self-contained: big token budget + errors count as
    undecided instead of crashing the run."""
    try:
        return gpt_generate(image, prompt, max_tokens)
    except Exception as e:
        print(f"  [api-fail] {type(e).__name__}")
        return ""

all_results["gpt-5.5-low"] = eval_model(
    gpt_generate_fixed, "gpt-5.5-low-fixed")

def norm(s):
    return re.sub(r"[^A-Z0-9]", "", s.upper())

READ_PROMPT = "What does the text label in this crop say?"

res = {}
correct = undecided = 0
for ex in stage5_pool:
    m = re.search(r"A: (.+)\nB: (.+)\nWhich", ex["prompt"])
    A, B = m.group(1), m.group(2)
    ans = norm(qwen_generate(ex["crop"], READ_PROMPT, 32))
    sa = difflib.SequenceMatcher(None, ans, norm(A)).ratio()
    sb = difflib.SequenceMatcher(None, ans, norm(B)).ratio()
    if sa == sb:
        undecided += 1
        continue
    correct += (sa > sb) == ex["a_is_true"]
n_dec = len(stage5_pool) - undecided
acc = round(100 * correct / n_dec, 1) if n_dec else None
res["5_ocr_tiebreak"] = {
    "acc_%": acc, "undecided": undecided, "n": len(stage5_pool)}

res["12_relation"] = all_results["qwen-domain-v2"]["12_relation"]

correct = undecided = 0
for ex in stage13_pool:
    p = ex["prompt"].replace(
        "Answer keep or remove.", "Answer yes or no.")
    ans = qwen_generate(ex["crop"], p, 32).lower()
    yes = ans.startswith("yes")
    no = ans.startswith("no")
    if not yes and not no:
        undecided += 1
        continue
    correct += (yes == ex["is_real"])
n_dec = len(stage13_pool) - undecided
acc = round(100 * correct / n_dec, 1) if n_dec else None
res["13_entity_val"] = {
    "acc_%": acc, "undecided": undecided, "n": len(stage13_pool)}

all_results["qwen-domain-v2-tolerant"] = res

models = list(all_results)
stages = ["5_ocr_tiebreak", "12_relation", "13_entity_val"]
header = f"{'stage':16s}"
for m in models:
    header += f"{m:>28s}"
header += "   baseline"
print(header)
print("-" * len(header))
for st in stages:
    line = f"{st:16s}"
    for m in models:
        r = all_results[m][st]
        a = r["acc_%"]
        cell = (str(a) + "%" if a is not None else "n/a")
        cell += f" (u{r['undecided']})"
        line += f"{cell:>28s}"
    line += "      50%"
    print(line)